# Ray Tune toy problem
Core idea -- instead of manually changing hyperparameters and re-tuning, define a search space, and Ray launches multiple trials automatically, each with different values sampled from that space.

In [1]:
!pip install ray[tune] -q

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.amp import GradScaler, autocast

from ray import tune
from ray.tune.schedulers import ASHAScheduler
import ray, os

# device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

In [6]:
ray.init(
    ignore_reinit_error=True,
    num_cpus=2,
    num_gpus=1,
)

print(ray.available_resources())


2026-06-25 12:40:02,912	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


{'node:__internal_head__': 1.0, 'CPU': 2.0, 'memory': 9239222272.0, 'node:172.28.0.12': 1.0, 'object_store_memory': 3959666688.0, 'GPU': 1.0, 'accelerator_type:T4': 1.0}


In [7]:
def train_cifar(config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ])

    train_set = datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
    val_set   = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=int(config["batch_size"]), shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_set,   batch_size=256,                       shuffle=False, num_workers=2)

    model     = SmallCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    scaler    = GradScaler(device, enabled=(device.type == 'cuda'))

    for epoch in range(10):
        # --- train ---
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss    = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()

        # --- validate ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                correct += outputs.argmax(1).eq(targets).sum().item()
                total   += targets.size(0)

        val_acc = correct / total

        # Report metric to Ray after each epoch. Ray uses these reported metrics to decide which trials to keep running and which to kill.
        tune.report({"val_acc": val_acc, "epoch": epoch + 1})

In [8]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN().to(device)

ray.init(ignore_reinit_error=True)

search_space = {
    "lr":           tune.loguniform(1e-4, 1e-2),   # sample LR on log scale
    "batch_size":   tune.choice([64, 128, 256]),
    "weight_decay": tune.loguniform(1e-5, 1e-3),
}

scheduler = ASHAScheduler(
    metric="val_acc",
    mode="max",
    max_t=10,           # max epochs per trial
    grace_period=3,     # let every trial run at least 3 epochs before killing
    reduction_factor=2  # kill bottom half of trials at each checkpoint
)

tuner = tune.Tuner(
    tune.with_resources(train_cifar, resources={"cpu": 1, "gpu": 1}),  # add this
    param_space=search_space,
    tune_config=tune.TuneConfig(
        num_samples=3,
        scheduler=ASHAScheduler(
            metric="val_acc",
            mode="max",
            max_t=3,
            grace_period=2,
            reduction_factor=2
        ),
    ),
)

# tuner = tune.Tuner(
#     train_cifar,
#     param_space=search_space,
#     tune_config=tune.TuneConfig(
#         num_samples=8,       # run 8 trials total
#         scheduler=scheduler,
#     ),
# )

results = tuner.fit()

2026-06-25 12:40:07,335	INFO worker.py:1828 -- Calling ray.init() again after it has already been called.


+--------------------------------------------------------------------+
| Configuration for experiment     train_cifar_2026-06-25_12-40-07   |
+--------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator             |
| Scheduler                        AsyncHyperBandScheduler           |
| Number of trials                 3                                 |
+--------------------------------------------------------------------+

View detailed results here: /root/ray_results/train_cifar_2026-06-25_12-40-07
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-06-25_12-39-19_103498_15308/artifacts/2026-06-25_12-40-07/train_cifar_2026-06-25_12-40-07/driver_artifacts`

Trial status: 3 PENDING
Current time: 2026-06-25 12:40:07. Total running time: 0s
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+--------------------------------------------------------------

  0%|          | 0.00/170M [00:00<?, ?B/s]
  0%|          | 65.5k/170M [00:00<05:18, 535kB/s]
  0%|          | 229k/170M [00:00<02:47, 1.02MB/s]
  1%|          | 918k/170M [00:00<00:54, 3.14MB/s]
  2%|▏         | 3.70M/170M [00:00<00:15, 10.9MB/s]
  6%|▌         | 9.80M/170M [00:00<00:06, 24.9MB/s]
  9%|▉         | 15.0M/170M [00:00<00:04, 32.9MB/s]
 11%|█▏        | 19.5M/170M [00:00<00:04, 36.5MB/s]
 15%|█▍        | 25.1M/170M [00:00<00:03, 42.2MB/s]
 17%|█▋        | 29.5M/170M [00:01<00:03, 42.6MB/s]
 20%|██        | 34.7M/170M [00:01<00:03, 45.2MB/s]
 23%|██▎       | 39.3M/170M [00:01<00:02, 45.3MB/s]
 26%|██▌       | 44.1M/170M [00:01<00:02, 46.2MB/s]
 29%|██▉       | 49.4M/170M [00:01<00:02, 48.2MB/s]
 32%|███▏      | 54.3M/170M [00:01<00:02, 48.1MB/s]
 35%|███▍      | 59.1M/170M [00:01<00:02, 48.2MB/s]
 38%|███▊      | 64.3M/170M [00:01<00:02, 49.4MB/s]
 41%|████      | 69.3M/170M [00:01<00:02, 48.0MB/s]
 44%|████▍     | 75.0M/170M [00:01<00:01, 49.9MB/s]
 47%|████▋     | 80.0M/1


Trial status: 1 RUNNING | 2 PENDING
Current time: 2026-06-25 12:40:37. Total running time: 30s
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+----------------------------------------------------------------------------------+
| Trial name                status              lr     batch_size     weight_decay |
+----------------------------------------------------------------------------------+
| train_cifar_ff2a4_00000   RUNNING    0.00344119              64      0.000163139 |
| train_cifar_ff2a4_00001   PENDING    0.000100593             64      0.000288086 |
| train_cifar_ff2a4_00002   PENDING    0.00571894             128      2.62713e-05 |
+----------------------------------------------------------------------------------+
Trial status: 1 RUNNING | 2 PENDING
Current time: 2026-06-25 12:41:07. Total running time: 1min 0s
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+-------------------------------------------------------

  0%|          | 0.00/170M [00:00<?, ?B/s]
  0%|          | 65.5k/170M [00:00<05:26, 522kB/s]
  0%|          | 229k/170M [00:00<02:51, 994kB/s] 
  0%|          | 655k/170M [00:00<01:13, 2.31MB/s]
  1%|          | 1.02M/170M [00:00<01:01, 2.75MB/s]
  1%|          | 1.47M/170M [00:00<00:50, 3.36MB/s]
  1%|          | 1.93M/170M [00:00<00:45, 3.69MB/s]
  1%|▏         | 2.42M/170M [00:00<00:41, 4.03MB/s]
  2%|▏         | 3.01M/170M [00:00<00:36, 4.54MB/s]
  2%|▏         | 3.60M/170M [00:00<00:33, 4.92MB/s]
  3%|▎         | 4.29M/170M [00:01<00:30, 5.39MB/s]
  3%|▎         | 5.01M/170M [00:01<00:27, 5.92MB/s]
  3%|▎         | 5.77M/170M [00:01<00:25, 6.35MB/s]
  4%|▍         | 6.68M/170M [00:01<00:23, 7.08MB/s]
  4%|▍         | 7.60M/170M [00:01<00:21, 7.68MB/s]
  5%|▌         | 8.55M/170M [00:01<00:19, 8.21MB/s]
  6%|▌         | 9.70M/170M [00:01<00:17, 9.17MB/s]
  6%|▋         | 10.8M/170M [00:01<00:16, 9.62MB/s]
  7%|▋         | 12.1M/170M [00:01<00:14, 10.6MB/s]
  8%|▊         | 13.5M/1

...

Trial status: 1 TERMINATED | 1 RUNNING | 1 PENDING
Current time: 2026-06-25 12:42:38. Total running time: 2min 30s
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+--------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                status                lr     batch_size     weight_decay     iter     total time (s)     val_acc     epoch |
+--------------------------------------------------------------------------------------------------------------------------------------+
| train_cifar_ff2a4_00001   RUNNING      0.000100593             64      0.000288086                                                   |
| train_cifar_ff2a4_00000   TERMINATED   0.00344119              64      0.000163139        3            95.9884      0.6073         3 |
| train_cifar_ff2a4_00002   PENDING      0.00571894             128      2.62713e-05                                       

  0%|          | 0.00/170M [00:00<?, ?B/s]
  0%|          | 32.8k/170M [00:00<10:23, 273kB/s]
  0%|          | 197k/170M [00:00<03:07, 906kB/s] 
  0%|          | 852k/170M [00:00<00:57, 2.95MB/s]
  2%|▏         | 3.44M/170M [00:00<00:16, 10.1MB/s]
  4%|▍         | 7.57M/170M [00:00<00:08, 18.7MB/s]
  7%|▋         | 11.7M/170M [00:00<00:06, 23.9MB/s]
  9%|▉         | 15.9M/170M [00:00<00:05, 27.0MB/s]
 12%|█▏        | 20.0M/170M [00:00<00:05, 29.1MB/s]
 14%|█▍        | 24.2M/170M [00:01<00:04, 30.6MB/s]
 17%|█▋        | 28.2M/170M [00:01<00:04, 33.2MB/s]
 19%|█▊        | 31.6M/170M [00:01<00:04, 32.6MB/s]
 20%|██        | 34.9M/170M [00:01<00:04, 31.7MB/s]
 23%|██▎       | 38.7M/170M [00:01<00:03, 33.0MB/s]
 25%|██▍       | 42.2M/170M [00:01<00:03, 33.5MB/s]
 27%|██▋       | 45.6M/170M [00:01<00:03, 32.9MB/s]
 29%|██▉       | 49.2M/170M [00:01<00:03, 33.0MB/s]
 31%|███       | 53.2M/170M [00:01<00:03, 35.1MB/s]
 33%|███▎      | 56.7M/170M [00:02<00:03, 33.9MB/s]
 35%|███▌      | 60.1M/1


Trial status: 2 TERMINATED | 1 RUNNING
Current time: 2026-06-25 12:44:08. Total running time: 4min 0s
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+--------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                status                lr     batch_size     weight_decay     iter     total time (s)     val_acc     epoch |
+--------------------------------------------------------------------------------------------------------------------------------------+
| train_cifar_ff2a4_00002   RUNNING      0.00571894             128      2.62713e-05                                                   |
| train_cifar_ff2a4_00000   TERMINATED   0.00344119              64      0.000163139        3            95.9884      0.6073         3 |
| train_cifar_ff2a4_00001   TERMINATED   0.000100593             64      0.000288086        3            95.5484      0.6155         3 

2026-06-25 12:44:48,786	INFO tune.py:1001 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_cifar_2026-06-25_12-40-07' in 0.0060s.



Trial train_cifar_ff2a4_00002 completed after 2 iterations at 2026-06-25 12:44:48. Total running time: 4min 41s
+--------------------------------------------------+
| Trial train_cifar_ff2a4_00002 result             |
+--------------------------------------------------+
| checkpoint_dir_name                              |
| time_this_iter_s                         26.1937 |
| time_total_s                             62.9293 |
| training_iteration                             2 |
| epoch                                          2 |
| val_acc                                   0.5527 |
+--------------------------------------------------+

Trial status: 3 TERMINATED
Current time: 2026-06-25 12:44:48. Total running time: 4min 41s
Logical resource usage: 1.0/2 CPUs, 1.0/1 GPUs (0.0/1.0 accelerator_type:T4)
+--------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                status                l

In [9]:
best = results.get_best_result(metric="val_acc", mode="max")

print("Best config:")
for k, v in best.config.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

print(f"\nBest val accuracy: {best.metrics['val_acc']*100:.1f}%")

# Show all trials sorted by val_acc
df = results.get_dataframe()
print("\nAll trials:")
print(df[["config/lr", "config/batch_size", "config/weight_decay", "val_acc"]]
      .sort_values("val_acc", ascending=False)
      .to_string(index=False))

Best config:
  lr: 0.000101
  batch_size: 64
  weight_decay: 0.000288

Best val accuracy: 61.6%

All trials:
 config/lr  config/batch_size  config/weight_decay  val_acc
  0.000101                 64             0.000288   0.6155
  0.003441                 64             0.000163   0.6073
  0.005719                128             0.000026   0.5527
